# RAG Complete Testing Notebook

This notebook provides end-to-end testing for the RAG system:

## Part 1: Document Processing & Chunking
- Load and process documents with configurable parameters
- Test different chunk sizes, embedding models
- Preview chunks before saving to database

## Part 2: Save to Database & Query Testing
- Delete old data for tenant
- Save processed chunks to PostgreSQL
- Test retrieval with different top-k values

## Part 3: LLM Generation Testing
- Build system prompt + context
- Test with different LLM providers (OpenRouter, Google, OpenAI)
- Compare results

**Note**: This is for testing only - does not modify production code.

---
# Setup & Configuration

In [1]:
# Required libraries
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
from docx import Document as DocxDocument
from sentence_transformers import SentenceTransformer
import uuid
import json
import numpy as np
from typing import List, Dict, Any
from pathlib import Path
import requests

print("✅ Libraries imported successfully")

C:\Users\gensh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported successfully


## Global Configuration

In [2]:
# ============================================
# DATABASE CONFIGURATION
# ============================================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "chatbot_itl",
    "user": "postgres",
    "password": "123456"
}

# ============================================
# TENANT SELECTION
# ============================================
TENANT_ID = "1193a40f-1d03-4ecd-a601-901a55589f56"  # <<< CHANGE THIS
COLLECTION_NAME = "knowledge_documents"

# ============================================
# DOCUMENT PATH
# ============================================
DOCUMENT_PATH = "../backend/eTMS.docx"  # <<< CHANGE THIS

print(f"✅ Configuration set:")
print(f"   Tenant ID: {TENANT_ID}")
print(f"   Document: {DOCUMENT_PATH}")
print(f"   Collection: {COLLECTION_NAME}")

✅ Configuration set:
   Tenant ID: 1193a40f-1d03-4ecd-a601-901a55589f56
   Document: ../backend/eTMS.docx
   Collection: knowledge_documents


---
# Part 1: Document Processing & Chunking

Test different chunking strategies and parameters before saving to database.

## 1.1 Processing Configuration

Configure chunking parameters here:

In [3]:
# ============================================
# CHUNKING CONFIGURATION (DYNAMIC)
# ============================================
CHUNK_SIZE = 400  # <<< CHANGE: 400, 600, 800, 1000
CHUNK_OVERLAP = 0  # <<< CHANGE: 0, 100, 200
CHUNK_METHOD = "character"  # <<< CHANGE: "character" or "recursive"

# ============================================
# EMBEDDING MODEL (DYNAMIC)
# ============================================
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # <<< CHANGE if needed
# Options:
# - "all-MiniLM-L6-v2" (384 dims, fast)
# - "paraphrase-multilingual-MiniLM-L12-v2" (384 dims, multilingual)
# - "all-mpnet-base-v2" (768 dims, better quality)

BATCH_SIZE = 64  # Embedding batch size

print(f"✅ Processing configuration:")
print(f"   Chunk size: {CHUNK_SIZE} characters")
print(f"   Chunk overlap: {CHUNK_OVERLAP} characters")
print(f"   Chunk method: {CHUNK_METHOD}")
print(f"   Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"   Batch size: {BATCH_SIZE}")

✅ Processing configuration:
   Chunk size: 400 characters
   Chunk overlap: 0 characters
   Chunk method: character
   Embedding model: all-MiniLM-L6-v2
   Batch size: 64


## 1.2 Load Document

In [5]:
# Check if document exists
doc_path = Path(DOCUMENT_PATH)
if not doc_path.exists():
    raise FileNotFoundError(f"Document not found: {DOCUMENT_PATH}")

# Load DOCX
print(f"Loading document: {DOCUMENT_PATH}")
doc = DocxDocument(DOCUMENT_PATH)
paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

print(f"✅ Loaded {len(paragraphs)} paragraphs")

# Extract headings for section tracking
headings = []
for para in doc.paragraphs:
    if para.style.name.startswith("Heading"):
        headings.append(para.text.strip())

print(f"✅ Found {len(headings)} headings")
print(f"\nFirst 5 headings:")
for i, h in enumerate(headings[:200], 1):
    print(f"  {i}. {h}")

Loading document: ../backend/eTMS.docx
✅ Loaded 4705 paragraphs
✅ Found 155 headings

First 5 headings:
  1. CÁC CHỨC NĂNG MỚI
  2. 1. Giới Thiệu Tổng Quan
  3. 2. Quy tắc và Các Thao Tác Cơ Bản Trên Hệ Thống
  4. 2.1. Quy Tắc Chung Trên Hệ Thống
  5. 2.2. Thuật ngữ và viết tắt
  6. 2.3. Thao Tác Chung
  7. 2.3.1. Import (Nhập dữ liệu)
  8. 2.3.2. Tìm Kiếm Dữ Liệu
  9. 2.3.3. Track and Trace
  10. 2.3.4. Đăng Nhập Vào Hệ Thống
  11. 3. Danh Mục (Catalogue)
  12. 3.1. Mạng lưới giao/nhận (Transport Network)
  13. 3.1.1. Địa điểm (Places)
  14. 3.1.2. Chiều dài tuyến đường (Distance Between Places)
  15. 3.1.3. Thông tin tuyến đường (Route Information)
  16. 3.1.4. Tuyến trung chuyển (Transit Route)
  17. 3.1.5. Hub
  18. 3.1.6. Chi nhánh (Branch)
  19. 3.1.7. Địa giới hành chính (Administrative Units)
  20. 3.1.8. Mã vùng (Zone Code)
  21. 3.1.9. Tuyến đường dự án (Route Project Information)
  22. 3.2. Danh sách đối tác (Partner)
  23. 3.2.1. Nhóm đối tác (Partner Group)
  24. 3.2.2. Da

## 1.3 Chunk Document

In [6]:
def chunk_text_character(text: str, size: int, overlap: int = 0) -> List[str]:
    """Split text by character count with optional overlap."""
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        
        # Move forward by (size - overlap)
        start = start + size - overlap
        
        # Prevent infinite loop
        if size <= overlap:
            break
    
    return chunks


def chunk_text_recursive(text: str, size: int, overlap: int = 0) -> List[str]:
    """Split text recursively by sentences/words (similar to RecursiveCharacterTextSplitter)."""
    # Simple implementation - split by sentence first, then by size
    import re
    
    # Split by sentence
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= size:
            current_chunk += sentence + " "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks


# Process chunks
print(f"Chunking with method: {CHUNK_METHOD}")
chunks = []
metadata_list = []

current_section = "Unknown"
for i, paragraph in enumerate(paragraphs):
    # Update current section if paragraph is a heading
    if paragraph in headings:
        current_section = paragraph
    
    # Chunk the paragraph
    if CHUNK_METHOD == "character":
        para_chunks = chunk_text_character(paragraph, CHUNK_SIZE, CHUNK_OVERLAP)
    else:
        para_chunks = chunk_text_recursive(paragraph, CHUNK_SIZE, CHUNK_OVERLAP)
    
    for chunk in para_chunks:
        page_number = i // 10 + 1  # Estimate page number
        
        chunks.append(chunk)
        metadata_list.append({
            "tenant_id": TENANT_ID,
            "document_name": Path(DOCUMENT_PATH).name,
            "page_number": page_number,
            "section_title": current_section,
            "chunk_size": len(chunk),
            "chunk_method": CHUNK_METHOD,
            "chunk_overlap": CHUNK_OVERLAP
        })

print(f"\n✅ Created {len(chunks)} chunks")
print(f"   Average chunk size: {sum(len(c) for c in chunks) // len(chunks)} characters")
print(f"   Min chunk size: {min(len(c) for c in chunks)} characters")
print(f"   Max chunk size: {max(len(c) for c in chunks)} characters")

Chunking with method: character

✅ Created 4736 chunks
   Average chunk size: 85 characters
   Min chunk size: 1 characters
   Max chunk size: 400 characters


## 1.4 Preview Chunks

Preview sample chunks to verify quality before saving to database.

In [7]:
# Show statistics by section
section_counts = {}
for meta in metadata_list:
    section = meta['section_title']
    section_counts[section] = section_counts.get(section, 0) + 1

print(f"📊 Top 10 sections by chunk count:")
for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"   {count:4d} chunks - {section[:80]}")

# Show sample chunks
print(f"\n📝 Sample chunks (first 5):")
for i in range(min(5, len(chunks))):
    print(f"\n{'='*100}")
    print(f"CHUNK {i+1}")
    print(f"{'='*100}")
    print(f"Section: {metadata_list[i]['section_title']}")
    print(f"Length: {len(chunks[i])} chars")
    print(f"Content:")
    print(f"-" * 100)
    print(chunks[i][:300] + ("..." if len(chunks[i]) > 300 else ""))
    print(f"\nMetadata: {metadata_list[i]}")

📊 Top 10 sections by chunk count:
    366 chunks - 4.10.2. LCL
    230 chunks - 5.5. Tạo đơn hàng dịch vụ thuê container, remooc.
    198 chunks - 4.11.1. Phụ phí/ Thu chi hộ FCL (FCL Surcharge/ Behalf)
    176 chunks - 4.10.1 FCL
    161 chunks - 2.3.4. Đăng Nhập Vào Hệ Thống
    110 chunks - 6.2 Quản lý SOA (giao diện mới)
    109 chunks - 4.2.2. LCL (LCL Buying)
    109 chunks - 7.2. Yêu Cầu Bảo Dưỡng Sửa chữa
    103 chunks - 3.3.1. Danh sách xe (Vehicle List)
    100 chunks - 3.2.2. Danh sách đối tác (Partner List)

📝 Sample chunks (first 5):

CHUNK 1
Section: Unknown
Length: 58 chars
Content:
----------------------------------------------------------------------------------------------------
TÀI LIỆU HỖ TRỢ ĐÀO TẠO VÀ HƯỚNG DẪN SỬ DỤNG PHẦN MỀM eTMS

Metadata: {'tenant_id': '1193a40f-1d03-4ecd-a601-901a55589f56', 'document_name': 'eTMS.docx', 'page_number': 1, 'section_title': 'Unknown', 'chunk_size': 58, 'chunk_method': 'character', 'chunk_overlap': 0}

CHUNK 2
Section: Unknown


## 1.5 Generate Embeddings

Generate embeddings for all chunks (but don't save yet).

In [ ]:
# Load embedding model
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
embedding_dim = embedder.get_sentence_embedding_dimension()

print(f"✅ Embedding model loaded")
print(f"   Model: {EMBEDDING_MODEL_NAME}")
print(f"   Dimensions: {embedding_dim}")

# Generate embeddings in batches
print(f"\nGenerating embeddings for {len(chunks)} chunks...")
all_embeddings = []
total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i+BATCH_SIZE]
    batch_embeddings = embedder.encode(batch)
    all_embeddings.extend(batch_embeddings)
    
    batch_num = i // BATCH_SIZE + 1
    if batch_num % 10 == 0 or batch_num == total_batches:
        print(f"   Processed batch {batch_num}/{total_batches}")

print(f"\n✅ Generated {len(all_embeddings)} embeddings")
print(f"   Embedding shape: {all_embeddings[0].shape}")

---
# Part 2: Save to Database & Query Testing

Save processed chunks to PostgreSQL and test retrieval.

## 2.1 Clear Old Data (Optional)

**WARNING**: This will delete all existing chunks for the tenant!

In [ ]:
# Set to True to delete old data
DELETE_OLD_DATA = False  # <<< CHANGE TO TRUE TO DELETE

if DELETE_OLD_DATA:
    print(f"⚠️  DELETING OLD DATA FOR TENANT: {TENANT_ID}")
    print("This action cannot be undone!")
    print()
    
    # Connect to database
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    
    # Get collection ID
    cur.execute("""
        SELECT uuid FROM langchain_pg_collection WHERE name = %s
    """, (COLLECTION_NAME,))
    collection = cur.fetchone()
    
    if collection:
        collection_id = collection['uuid']
        
        # Count before delete
        cur.execute("""
            SELECT COUNT(*) as count
            FROM langchain_pg_embedding
            WHERE collection_id = %s
            AND cmetadata->>'tenant_id' = %s
        """, (collection_id, TENANT_ID))
        before_count = cur.fetchone()['count']
        print(f"Chunks before delete: {before_count}")
        
        # Delete
        cur.execute("""
            DELETE FROM langchain_pg_embedding
            WHERE collection_id = %s
            AND cmetadata->>'tenant_id' = %s
        """, (collection_id, TENANT_ID))
        conn.commit()
        
        deleted_count = cur.rowcount
        print(f"✅ Deleted {deleted_count} chunks")
    else:
        print(f"⚠️  Collection '{COLLECTION_NAME}' not found")
    
    cur.close()
    conn.close()
else:
    print("ℹ️  Skipping deletion (DELETE_OLD_DATA = False)")
    print("   Set DELETE_OLD_DATA = True to delete old data")

## 2.2 Save Chunks to Database

In [ ]:
# Set to True to save to database
SAVE_TO_DATABASE = False  # <<< CHANGE TO TRUE TO SAVE

if SAVE_TO_DATABASE:
    print(f"💾 SAVING {len(chunks)} CHUNKS TO DATABASE")
    print(f"   Tenant ID: {TENANT_ID}")
    print(f"   Collection: {COLLECTION_NAME}")
    print()
    
    # Connect to database
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    
    # Get or create collection
    cur.execute("""
        SELECT uuid FROM langchain_pg_collection WHERE name = %s
    """, (COLLECTION_NAME,))
    collection = cur.fetchone()
    
    if collection:
        collection_id = collection['uuid']
        print(f"✅ Using existing collection: {collection_id}")
    else:
        collection_id = uuid.uuid4()
        cur.execute("""
            INSERT INTO langchain_pg_collection (uuid, name, cmetadata)
            VALUES (%s, %s, %s)
        """, (collection_id, COLLECTION_NAME, json.dumps({})))
        conn.commit()
        print(f"✅ Created new collection: {collection_id}")
    
    # Prepare data for batch insert
    print(f"\nPreparing data for insertion...")
    data_to_insert = []
    for i, (chunk, embedding, metadata) in enumerate(zip(chunks, all_embeddings, metadata_list)):
        chunk_id = str(uuid.uuid4())
        embedding_list = embedding.tolist()
        metadata["chunk_index"] = i
        
        data_to_insert.append((
            chunk_id,
            collection_id,
            embedding_list,
            chunk,
            json.dumps(metadata)
        ))
    
    # Batch insert
    print(f"Inserting {len(data_to_insert)} chunks...")
    total_batches = (len(data_to_insert) + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(0, len(data_to_insert), BATCH_SIZE):
        batch = data_to_insert[i:i+BATCH_SIZE]
        execute_batch(
            cur,
            """
            INSERT INTO langchain_pg_embedding (
                id, collection_id, embedding, document, cmetadata
            ) VALUES (%s, %s, %s::vector, %s, %s::jsonb)
            """,
            batch
        )
        conn.commit()
        
        batch_num = i // BATCH_SIZE + 1
        if batch_num % 10 == 0 or batch_num == total_batches:
            print(f"   Saved batch {batch_num}/{total_batches}")
    
    print(f"\n✅ Successfully saved {len(data_to_insert)} chunks to database")
    
    cur.close()
    conn.close()
else:
    print("ℹ️  Skipping database save (SAVE_TO_DATABASE = False)")
    print("   Set SAVE_TO_DATABASE = True to save to database")

## 2.3 Query Configuration

In [ ]:
# ============================================
# QUERY CONFIGURATION
# ============================================
TEST_QUERY = "Hướng dẫn tạo mới bảng báo giá bán LCL?"  # <<< CHANGE THIS
TOP_K = 3  # <<< CHANGE: Number of chunks to retrieve

print(f"✅ Query configuration:")
print(f"   Query: {TEST_QUERY}")
print(f"   Top-K: {TOP_K}")

## 2.4 Test Retrieval

In [ ]:
def retrieve_chunks(query: str, tenant_id: str, top_k: int = 3) -> List[Dict]:
    """Retrieve relevant chunks from database."""
    # Connect to database
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    
    # Get collection ID
    cur.execute("""
        SELECT uuid FROM langchain_pg_collection WHERE name = %s
    """, (COLLECTION_NAME,))
    collection = cur.fetchone()
    
    if not collection:
        raise ValueError(f"Collection '{COLLECTION_NAME}' not found")
    
    collection_id = collection['uuid']
    
    # Generate query embedding
    query_embedding = embedder.encode(query).tolist()
    
    # Search for similar chunks
    cur.execute("""
        SELECT
            document as content,
            cmetadata as metadata,
            embedding <=> %s::vector as distance
        FROM langchain_pg_embedding
        WHERE collection_id = %s
        AND cmetadata->>'tenant_id' = %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (query_embedding, collection_id, tenant_id, query_embedding, top_k))
    
    results = cur.fetchall()
    
    cur.close()
    conn.close()
    
    return results


# Execute retrieval
print(f"\n{'='*100}")
print(f"RETRIEVAL TEST")
print(f"{'='*100}")
print(f"Query: {TEST_QUERY}")
print(f"Top-K: {TOP_K}")
print()

retrieved_chunks = retrieve_chunks(TEST_QUERY, TENANT_ID, TOP_K)

print(f"✅ Retrieved {len(retrieved_chunks)} chunks\n")

# Display each chunk
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{'='*100}")
    print(f"CHUNK {i}/{len(retrieved_chunks)}")
    print(f"{'='*100}")
    print(f"Distance (cosine): {chunk['distance']:.4f}")
    print(f"Section: {chunk['metadata'].get('section_title', 'Unknown')}")
    print(f"Page: {chunk['metadata'].get('page_number', 'Unknown')}")
    print(f"Length: {len(chunk['content'])} characters")
    print(f"\nContent:")
    print(f"-" * 100)
    print(chunk['content'])
    print()

---
# Part 3: LLM Generation Testing

Build system prompt + context and test with different LLM providers.

## 3.1 Build Context for LLM

In [ ]:
# Combine retrieved chunks into context
context_parts = []
for i, chunk in enumerate(retrieved_chunks, 1):
    section = chunk['metadata'].get('section_title', 'Unknown')
    context_parts.append(f"[Document {i} - Section: {section}]\n{chunk['content']}")

combined_context = "\n\n".join(context_parts)

print(f"{'='*100}")
print(f"COMBINED CONTEXT (Retrieved Documents)")
print(f"{'='*100}")
print(f"Total length: {len(combined_context)} characters")
print(f"Number of chunks: {len(retrieved_chunks)}")
print()
print(combined_context)

## 3.2 Build System Prompt

This simulates the prompt used in `domain_agents.py`

In [ ]:
# System prompt (similar to AgentAnalysis in domain_agents.py)
SYSTEM_PROMPT = """Bạn là trợ lý AI chuyên nghiệp, hỗ trợ người dùng trả lời câu hỏi về hệ thống eTMS.

QUAN TRỌNG:
1. Sử dụng thông tin từ tài liệu được cung cấp để trả lời
2. Trả lời chi tiết, rõ ràng, có cấu trúc
3. Nếu câu hỏi liên quan đến quy trình, liệt kê đầy đủ các bước
4. Nếu không tìm thấy thông tin, hãy nói rõ điều đó
5. Trích dẫn nguồn khi cần thiết [Source: Section Name]

Thông tin tham khảo từ tài liệu:
{context}
"""

# Build full prompt
full_system_prompt = SYSTEM_PROMPT.format(context=combined_context)

print(f"{'='*100}")
print(f"SYSTEM PROMPT (To send to LLM)")
print(f"{'='*100}")
print(f"Length: {len(full_system_prompt)} characters")
print()
print(full_system_prompt)

## 3.3 LLM Provider Configuration

In [ ]:
# ============================================
# LLM PROVIDER CONFIGURATION
# ============================================

# OPTION 1: OpenRouter
OPENROUTER_API_KEY = ""  # <<< ENTER YOUR KEY
OPENROUTER_MODEL = "anthropic/claude-3.5-sonnet"

# OPTION 2: Google AI (Gemini)
GOOGLE_API_KEY = ""  # <<< ENTER YOUR KEY
GOOGLE_MODEL = "gemini-1.5-pro"

# OPTION 3: OpenAI
OPENAI_API_KEY = ""  # <<< ENTER YOUR KEY
OPENAI_MODEL = "gpt-4-turbo"

# Choose which provider to use
USE_PROVIDER = "openrouter"  # <<< CHANGE: "openrouter", "google", "openai"

print(f"✅ LLM Provider selected: {USE_PROVIDER.upper()}")

## 3.4 LLM Calling Functions

In [ ]:
def call_openrouter(query: str, system_prompt: str, api_key: str, model: str) -> str:
    """Call OpenRouter API (compatible with LangChain format)."""
    url = "https://openrouter.ai/api/v1/chat/completions"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result['choices'][0]['message']['content']


def call_google_ai(query: str, system_prompt: str, api_key: str, model: str) -> str:
    """Call Google AI API (Gemini)."""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
    
    # Combine system prompt and query for Gemini
    full_prompt = f"{system_prompt}\n\nCâu hỏi: {query}"
    
    headers = {
        "Content-Type": "application/json"
    }
    
    payload = {
        "contents": [
            {
                "parts": [
                    {"text": full_prompt}
                ]
            }
        ]
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result['candidates'][0]['content']['parts'][0]['text']


def call_openai(query: str, system_prompt: str, api_key: str, model: str) -> str:
    """Call OpenAI API."""
    url = "https://api.openai.com/v1/chat/completions"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result['choices'][0]['message']['content']


print("✅ LLM calling functions defined")

## 3.5 Call LLM and Display Result

In [ ]:
# Call the selected LLM
print(f"\n{'='*100}")
print(f"LLM GENERATION TEST")
print(f"{'='*100}")
print(f"Provider: {USE_PROVIDER.upper()}")
print(f"Query: {TEST_QUERY}")
print(f"System prompt length: {len(full_system_prompt)} characters")
print(f"Context length: {len(combined_context)} characters")
print()

try:
    if USE_PROVIDER == "openrouter":
        if not OPENROUTER_API_KEY:
            raise ValueError("Please set OPENROUTER_API_KEY")
        print(f"Model: {OPENROUTER_MODEL}")
        print("Calling OpenRouter API...\n")
        response = call_openrouter(TEST_QUERY, full_system_prompt, OPENROUTER_API_KEY, OPENROUTER_MODEL)
        
    elif USE_PROVIDER == "google":
        if not GOOGLE_API_KEY:
            raise ValueError("Please set GOOGLE_API_KEY")
        print(f"Model: {GOOGLE_MODEL}")
        print("Calling Google AI API...\n")
        response = call_google_ai(TEST_QUERY, full_system_prompt, GOOGLE_API_KEY, GOOGLE_MODEL)
        
    elif USE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise ValueError("Please set OPENAI_API_KEY")
        print(f"Model: {OPENAI_MODEL}")
        print("Calling OpenAI API...\n")
        response = call_openai(TEST_QUERY, full_system_prompt, OPENAI_API_KEY, OPENAI_MODEL)
        
    else:
        raise ValueError(f"Unknown provider: {USE_PROVIDER}")
    
    print(f"{'='*100}")
    print(f"LLM RESPONSE")
    print(f"{'='*100}")
    print(response)
    print()
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

---
# Summary

## Workflow Summary:

### Part 1: Document Processing
1. Configure chunk size, overlap, method, embedding model
2. Load DOCX document
3. Chunk text with configurable parameters
4. Preview chunks and statistics
5. Generate embeddings

### Part 2: Database Operations
1. Optionally clear old data
2. Save chunks + embeddings to PostgreSQL
3. Test retrieval with different top-k values
4. View retrieved chunks

### Part 3: LLM Testing
1. Build context from retrieved chunks
2. Build system prompt (similar to domain_agents.py)
3. Test with OpenRouter/Google/OpenAI
4. Compare results

## Next Steps:
- If results are good → Use these parameters in production code
- If results are poor → Adjust chunk_size, top_k, or embedding model and re-test
- Compare different chunking strategies and LLM providers